# 03_00 — Model Training: Feature Selection

**Pipeline:** `model_training`  
**Kedro nodes:** `build_training_data_node` → `train_isolation_forest_node` → `evaluate_model_node`  
**Input catalog:** `prepared_flights` (data/04_feature/)  
**Output catalog:** `isolation_forest`, `model_metrics`

---

## Contexto

Com os dados preparados, esta etapa treina um detector de anomalias não-supervisionado.
O objetivo é detectar a **falha do motor** o mais rápido possível após ela ocorrer.

### Por que Isolation Forest?

- Não requer dados rotulados para treinar (apenas o parâmetro `contamination`)
- Funciona bem com dados de alta dimensão e temporal
- Produz um **anomaly score** contínuo, não só uma classificação binária

### Fluxo da pipeline
1. **Skip inicial**: remove a fase de transição de altitude no início de cada voo
2. **Seleção de features**: Cohen's d ranqueia features por separabilidade fault vs normal
3. **Sliding windows**: cada amostra = últimos `window_size` timesteps achatados
4. **Treino**: Isolation Forest com split temporal 70/30
5. **Avaliação**: latência de detecção = tempo entre falha real e primeiro alerta

> **Para rodar toda a pipeline no Kedro:** `kedro run --pipeline=model_training`

## Imports e parâmetros

In [ ]:
import yaml
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from aeroespacial_2.pipelines.model_training.nodes import (
    create_windows,
    evaluate_model,
    select_top_features_effect_size,
    split_windows,
    train_isolation_forest,
)

_params = yaml.safe_load(
    (Path("../conf/base/parameters.yml")).read_text()
)["model_training"]

PREPARED_FILE  = "../data/04_feature/carbonZ_2018-07-18-15-53-31_1_engine_failure.csv"
WINDOW_SIZE    = _params["window_size"]
N_TOP_FEATURES = _params["n_top_features"]
CONTAMINATION  = _params["contamination"]
N_ESTIMATORS   = _params["n_estimators"]
TRAIN_RATIO    = _params["train_ratio"]
TARGET_COL     = _params["target_col"]
TIMESTAMP_COL  = _params["timestamp_col"]
SKIP_SECONDS   = _params.get("skip_seconds", 0.0)

print("Parâmetros carregados de conf/base/parameters.yml:")
for k, v in _params.items():
    print(f"  {k}: {v}")

In [ ]:
df_raw = pd.read_csv(PREPARED_FILE)

# Remove a fase de transição inicial (aeronave descendo para altitude de cruise)
df = df_raw[df_raw[TIMESTAMP_COL] >= SKIP_SECONDS].copy() if SKIP_SECONDS > 0 else df_raw.copy()

print(f"Shape original: {df_raw.shape}  →  após skip {SKIP_SECONDS}s: {df.shape}")
print(f"Falhas: {df[TARGET_COL].sum():.0f} amostras ({df[TARGET_COL].mean()*100:.1f}%)")
df.head(3)

## Passo 1 — Seleção de features com Cohen's d

Ranqueamos cada feature pelo **effect size de Cohen's d** entre as distribuições
de voo normal e de falha:

$$d = \\frac{|\\mu_{\\text{falha}} - \\mu_{\\text{normal}}|}{\\sigma_{\\text{pooled}}}$$

Isso responde diretamente à pergunta do Isolation Forest: **"qual feature tem distribuição
mais diferente entre estado normal e estado de falha?"**

Vantagens sobre Random Forest:
- **Não depende da estrutura temporal**: mede separabilidade de distribuições, não correlação ponto a ponto com o label
- **Não cria correlações espúrias**: o RF com label adiantado selecionava features de vento/heading que variavam na janela de tempo pré-falha por coincidência
- **Robusto ao desbalanceamento**: a normalização pelo σ_pooled ajusta automaticamente pela variabilidade intra-grupo

In [ ]:
features = [c for c in df.columns if c not in [TARGET_COL, TIMESTAMP_COL]]
top_features = select_top_features_effect_size(df, features, TARGET_COL, n_top=N_TOP_FEATURES)

print(f"Top {N_TOP_FEATURES} features (Cohen's d):")
for i, f in enumerate(top_features, 1):
    print(f"  {i:2d}. {f}")

In [ ]:
# Calcula Cohen's d para todas as features e visualiza as selecionadas
normal_grp = df.loc[df[TARGET_COL] == 0, features]
fault_grp  = df.loc[df[TARGET_COL] == 1, features]

n0, n1 = len(normal_grp), len(fault_grp)
mean_diff  = (fault_grp.mean() - normal_grp.mean()).abs()
pooled_std = np.sqrt(
    ((n0 - 1) * normal_grp.std() ** 2 + (n1 - 1) * fault_grp.std() ** 2) / (n0 + n1 - 2)
).replace(0, np.nan)
cohens_d = (mean_diff / pooled_std).fillna(0)

top_scores = cohens_d[top_features].sort_values(ascending=True)

# Convenção: d < 0.2 pequeno | 0.2–0.8 médio | 0.8–1.2 grande | > 1.2 muito grande
def d_color(d):
    if d >= 1.2: return "tomato"
    if d >= 0.8: return "steelblue"
    return "lightsteelblue"

colors = [d_color(v) for v in top_scores.values]

fig, ax = plt.subplots(figsize=(10, N_TOP_FEATURES * 0.5 + 1))
ax.barh(top_scores.index, top_scores.values, color=colors)
ax.axvline(0.8, color="steelblue", linestyle="--", linewidth=1.2, label="d=0.8 (efeito grande)")
ax.axvline(1.2, color="tomato",    linestyle="--", linewidth=1.2, label="d=1.2 (efeito muito grande)")
ax.set_xlabel("Cohen's d  (|μ_falha − μ_normal| / σ_pooled)")
ax.set_title(f"Separabilidade fault vs normal — top {N_TOP_FEATURES} features por Cohen's d")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis="x")

for i, (feat, val) in enumerate(zip(top_scores.index, top_scores.values)):
    ax.text(val + 0.02, i, f"{val:.2f}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

# Features com efeito muito grande
muito_grandes = top_scores[top_scores >= 1.2]
print(f"Features com d ≥ 1.2 (efeito muito grande): {len(muito_grandes)}")
for f, d in muito_grandes.sort_values(ascending=False).items():
    print(f"  {f}: d={d:.2f}")

## Passo 2 — Sliding windows

Cada amostra é o achatamento dos últimos `window_size` timesteps.
A 100 Hz, `window_size=20` captura 0.2 segundos de contexto temporal.

In [ ]:
X, y = create_windows(df, WINDOW_SIZE, top_features, TARGET_COL)
print(f"X shape: {X.shape}  (n_amostras, window_size × n_features)")
print(f"y shape: {y.shape}")
print(f"Positivos (falha): {y.sum()} ({y.mean()*100:.1f}%)")

## Passo 3 — Split temporal e treino

O split é **temporal** (sem shuffle): os primeiros 70% para treino, os últimos 30% para teste.
O embaralhamento vazaria informações sobre a falha futura para o treino.

In [ ]:
X_train, X_test, y_train, y_test = split_windows(X, y, train_ratio=TRAIN_RATIO)
timestamps = df[TIMESTAMP_COL].values[WINDOW_SIZE:]
split_idx = int(len(timestamps) * TRAIN_RATIO)
ts_test = timestamps[split_idx:]

print(f"Train: {X_train.shape} | {y_train.sum():.0f} falhas")
print(f"Test:  {X_test.shape}  | {y_test.sum():.0f} falhas")

In [ ]:
model = train_isolation_forest(X_train, contamination=CONTAMINATION, n_estimators=N_ESTIMATORS)
print("Modelo treinado!")

## Passo 4 — Avaliação

In [ ]:
metrics = evaluate_model(model, X_test, y_test, ts_test)
print(f"Falha real:     {metrics.get('real_fault_time_s', 'N/A'):.2f}s")
print(f"Detecao:        {metrics.get('predicted_fault_time_s', 'N/A'):.2f}s")
print(f"Latencia:       {metrics.get('detection_latency_s', 'N/A'):.2f}s")

In [ ]:
# Predicoes e scores
raw_preds = model.predict(X_test)
y_pred = np.where(raw_preds == -1, 1, 0)
scores = model.decision_function(X_test)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Painel 1: falha real vs detectada
axes[0].plot(ts_test, y_test, color="red", alpha=0.6, linewidth=2, label="Falha real")
axes[0].plot(ts_test, y_pred, color="blue", linestyle="--", linewidth=1, label="Detecção")
axes[0].set_ylabel("Estado (0=normal, 1=falha)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_title("Isolation Forest — Falha Real vs Detecção")

# Painel 2: anomaly score
axes[1].plot(ts_test, scores, color="purple", linewidth=0.8, label="Anomaly Score")
axes[1].axhline(0, color="red", linestyle="--", linewidth=1.5, label="Limiar")
if metrics.get("real_fault_time_s"):
    axes[1].axvline(metrics["real_fault_time_s"], color="black", linewidth=1.5, label="Falha real")
axes[1].set_ylabel("Score (negativo = mais anômalo)")
axes[1].set_xlabel("Tempo (s)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_title('"Quão suspeito" o modelo acha o voo')

plt.tight_layout()
plt.show()

---
**Próximo:** `03_01_model2.ipynb` → tuning do parâmetro `contamination` para otimizar a latência de detecção.